# Securing Couchbase MCP Server with Keycloak — Browser Login Flows

This tutorial covers two browser-based OAuth flows for the Couchbase MCP server using Keycloak as the identity provider:

1. **Non-DCR flow** — pre-registered client with browser login, tested with MCP Inspector and VS Code
2. **DCR flow** — dynamic client registration with browser login, tested with MCP Inspector

---

## Prerequisites

- Docker installed (to run Keycloak locally)
- Couchbase MCP server installed via PyPI (`uvx couchbase-mcp-server`)
- For non-DCR flow: MCP Inspector (`npx @modelcontextprotocol/inspector`) or any IDE that supports non-DCR flow
- For DCR flow: MCP Inspector or any IDE that supports DCR flow
- A running Couchbase cluster with credentials

---

## Part 1 — Run Keycloak Locally

```bash
docker run -p 8080:8080 \
  -e KC_BOOTSTRAP_ADMIN_USERNAME=admin \
  -e KC_BOOTSTRAP_ADMIN_PASSWORD=admin \
  quay.io/keycloak/keycloak:latest start-dev
```

Open `http://localhost:8080` and log in with `admin` / `admin`.

<img src="keycloak_screenshots/key_1.png" width="500">

---

## Part 2 — Keycloak Setup (shared by both flows)

### Step 2.1 — Create a Realm

1. Click the top-left dropdown → **Create Realm**
2. Name it `mcp-realm` → **Create**

<img src="keycloak_screenshots/key_2.png" width="500">

### Step 2.2 — Create custom scopes

1. Left nav → **Client Scopes** → **Create client scope**
2. Create `couchbase-mcp:read`:
   - **Name**: `couchbase-mcp:read`
   - **Type**: Optional
   - **Protocol**: openid-connect
   - **Include in token scope** → toggle **On** — this makes the scope name appear in the `scope` claim of the token. Without this, the scope is assigned but never shows up in the token even if the client requests it.
   - **Save**
3. Repeat for `couchbase-mcp:write`

<img src="keycloak_screenshots/key_3.png" width="500">

### Step 2.3 — Create the MCP Server client

This represents the resource server (the audience).

1. Left nav → **Clients** → **Create client**
2. **Client ID**: `couchbase-mcp-server`
3. **Client authentication**: Off (public)
4. Click through to **Save**
5. Go to **Client Scopes** tab → **Add client scope** → add both `couchbase-mcp:read` and `couchbase-mcp:write` as **Optional**

<img src="keycloak_screenshots/key_4.png" width="500">

<img src="keycloak_screenshots/key_5.png" width="500">

<img src="keycloak_screenshots/key_6.png" width="500">

### Step 2.4 — Add audience mapper

The MCP server checks the `aud` claim. You need to make Keycloak put `couchbase-mcp-server` in the token audience.

1. Go to **Client Scopes** → `couchbase-mcp:read` → **Mappers** tab → **Add mapper** → **By configuration** → **Audience**
2. Fill in:
   - **Name**: `couchbase-mcp-audience`
   - **Included Client Audience**: `couchbase-mcp-server`
   - **Add to access token**: On
3. **Save**
4. Repeat on `couchbase-mcp:write` scope

<img src="keycloak_screenshots/key_10.png" width="500">

### Step 2.5 — Create a test user

1. Left nav → **Users** → **Add user**
2. Fill in:
   - **Username**: `testuser`
   - **First name**: `Test`
   - **Last name**: `User`
   - **Email**: `testuser@test.com`
3. **Details** tab → toggle **Email verified** → **On** → **Save** — without this the login flow returns `Account is not fully set up`
4. **Credentials** tab → **Set password** → `password` → turn off **Temporary** → **Save**

> ⚠️ First name, last name, and email must be filled in — Keycloak blocks login if the user profile is incomplete.

<img src="keycloak_screenshots/key_11.png" width="500">

<img src="keycloak_screenshots/key_7.png" width="500">


### Step 2.6 — Create the pre-registered client (for Non-DCR)

1. Left nav → **Clients** → **Create client**
2. **Client ID**: `mcp-inspector-client`
3. **Client authentication**: Off (public)
4. **Authentication flow**: check **Standard flow** only
5. **Valid redirect URIs** — add all of the following:
   - `http://localhost:6274/oauth/callback` — MCP Inspector
   - `http://127.0.0.1:6274/oauth/callback` — MCP Inspector (alternate)
   - `http://127.0.0.1:33418` — VS Code
   - `https://vscode.dev/redirect` — VS Code (web)
6. **Web origins**:
   - `http://localhost:6274`
   - `http://127.0.0.1:6274`
7. **Save**
8. **Client Scopes** tab → add `couchbase-mcp:read` and `couchbase-mcp:write` as Optional


<img src="keycloak_screenshots/key_15.png" width="500">

<img src="keycloak_screenshots/key_22.png" width="500">

<img src="keycloak_screenshots/key_16.png" width="500">

### Step 2.7 — Collect server config values

| Value | Format |
|---|---|
| **Issuer** | `http://localhost:8080/realms/mcp-realm` |
| **JWKS URI** | `http://localhost:8080/realms/mcp-realm/protocol/openid-connect/certs` |
| **Audience** | `couchbase-mcp-server` |

---

## Part 3 — Non-DCR Flow (Pre-registered Client)

This flow uses a pre-registered public client (`mcp-inspector-client`) with a browser-based authorization code + PKCE flow. The client ID must be provided in the Inspector auth settings.

### Step 3.1 — Start the MCP server with PRM

```bash
uvx couchbase-mcp-server \
  --transport=http \
  --connection-string="couchbase://127.0.0.1" \
  --username="Administrator" \
  --password="<your-couchbase-password>" \
  --read-only-mode=false \
  --oauth-jwks-uri="http://localhost:8080/realms/mcp-realm/protocol/openid-connect/certs" \
  --oauth-issuer="http://localhost:8080/realms/mcp-realm" \
  --oauth-audience="couchbase-mcp-server" \
  --oauth-mcp-base-url="http://127.0.0.1:8000"
```

**Verify the PRM document:**

```bash
curl -s http://127.0.0.1:8000/.well-known/oauth-protected-resource/mcp | python3 -m json.tool
```

Expected response:

```json
{
  "resource": "http://127.0.0.1:8000/mcp",
  "authorization_servers": ["http://localhost:8080/realms/mcp-realm"],
  "scopes_supported": ["couchbase-mcp:read", "couchbase-mcp:write"]
}
```


### Step 3.2 — Configure MCP Inspector

```bash
npx @modelcontextprotocol/inspector
```

In the Inspector UI:

| Field | Value |
|---|---|
| **Transport Type** | Streamable HTTP |
| **URL** | `http://127.0.0.1:8000/mcp` |
| **Client ID** | `mcp-inspector-client` |
| **Client Secret** | *(leave blank)* |
| **Redirect URL** | `http://localhost:6274/oauth/callback` |
| **Scope** | `openid couchbase-mcp:read couchbase-mcp:write` |

Authorization URL and Token URL are auto-discovered from the PRM document.


### Step 3.3 — Connect and verify via MCP Inspector

1. Click **Connect**
2. A browser window opens with the Keycloak login page
3. Log in with `testuser` / `password`
4. Inspector shows **"Successfully authenticated with OAuth"**


<img src="keycloak_screenshots/key_17.png" width="500">

<img src="keycloak_screenshots/key_18.png" width="500">

**Verify:**
1. **Tools** tab → **List Tools** — all 24 tools visible
2. Run a read tool → success
3. Run a write tool → success


### Step 3.4 — Connect via VS Code

In VS Code, add the MCP server to `mcp.json` with no token — VS Code will prompt for the client ID and drive the auth code + PKCE flow automatically:

```json
{
  "servers": {
    "couchbase-keycloak-nondcr": {
      "type": "http",
      "url": "http://127.0.0.1:8000/mcp"
    }
  }
}
```

When VS Code prompts for a **Client ID**, enter `mcp-inspector-client`. A browser window will open with the Keycloak login page → log in with `testuser` / `password` → VS Code connects.


<img src="keycloak_screenshots/key_19.png" width="500">

<img src="keycloak_screenshots/key_20.png" width="500">

<img src="keycloak_screenshots/key_21.png" width="500">

<img src="keycloak_screenshots/key_24.png" width="500">

---

## Part 4 — DCR Flow (Dynamic Client Registration)

This flow requires no pre-registered client ID. MCP Inspector discovers Keycloak via the PRM document and self-registers anonymously.

### Step 4.1 — Enable anonymous DCR in Keycloak

By default Keycloak locks down DCR. You need to configure it to allow anonymous registration.

1. Make sure you're in `mcp-realm`
2. Left nav → **Clients** → **Client registration** tab → **Client registration policies** sub-tab
3. Under **Anonymous Access Policies**, find **Client Disabled Policy** → **Delete** it — it auto-disables newly registered clients which breaks the flow


4. Under **Anonymous Access Policies**, find **Trusted Hosts** → **Delete** it. 

> ℹ️ **If you want to restrict which hosts can register clients** (e.g. in a production or shared environment), keep the Trusted Hosts policy instead of deleting it and add the specific hosts: ex: `localhost`, `127.0.0.1`, `192.168.65.1` , and any remote hosts. Check **Events** in Keycloak for `CLIENT_REGISTER_ERROR` entries to see exactly which IP is being rejected if a client fails to register.

<img src="keycloak_screenshots/key_36.png" width="500">

5. **Trusted redirect URIs** — leave empty to allow any redirect URI, or if you want to restrict to specific clients:
   - `http://localhost:6274/oauth/callback` — MCP Inspector
   - `http://127.0.0.1:6274/oauth/callback` — MCP Inspector (alternate)
   - `http://127.0.0.1:33418` — VS Code
   - `https://vscode.dev/redirect` — VS Code (web)


### Step 4.2 — Verify DCR is working

Before connecting with Inspector, confirm the DCR endpoint accepts anonymous registration:

```bash
curl -X POST \
  http://127.0.0.1:8080/realms/mcp-realm/clients-registrations/openid-connect \
  -H "Content-Type: application/json" \
  -d '{
    "client_name": "test-dcr-client",
    "redirect_uris": ["http://127.0.0.1:6274/oauth/callback"],
    "grant_types": ["authorization_code"],
    "response_types": ["code"],
    "token_endpoint_auth_method": "none",
    "scope": "openid couchbase-mcp:read couchbase-mcp:write"
  }'
```

If you get back a JSON with a `client_id` → DCR is working.

> ℹ️ **No `scope` in the DCR request** — Keycloak automatically assigns the default scopes (including `couchbase-mcp:read` and `couchbase-mcp:write`) to DCR clients because **Allow Default Scopes** is On. If you explicitly pass `scope` in the DCR payload, the Allowed Client Scopes policy will reject it. Leave scope out and let Keycloak assign it automatically.


### Step 4.3 — Start the MCP server

Same server command as Non-DCR:

```bash
uvx couchbase-mcp-server \
  --transport=http \
  --connection-string="couchbase://127.0.0.1" \
  --username="Administrator" \
  --password="<your-couchbase-password>" \
  --read-only-mode=false \
  --oauth-jwks-uri="http://localhost:8080/realms/mcp-realm/protocol/openid-connect/certs" \
  --oauth-issuer="http://localhost:8080/realms/mcp-realm" \
  --oauth-audience="couchbase-mcp-server" \
  --oauth-mcp-base-url="http://127.0.0.1:8000"
```

### Step 4.4 — Configure MCP Inspector for DCR

```bash
npx @modelcontextprotocol/inspector
```

In the Inspector UI:

| Field | Value |
|---|---|
| **Transport Type** | Streamable HTTP |
| **URL** | `http://127.0.0.1:8000/mcp` |
| **Client ID** | *(leave blank)* |
| **Client Secret** | *(leave blank)* |
| **Redirect URL** | `http://localhost:6274/oauth/callback` |
| **Scope** | `openid couchbase-mcp:read couchbase-mcp:write` |

Inspector will hit the PRM, discover Keycloak, call the DCR endpoint anonymously, get a `client_id`, then drive the auth code + PKCE flow automatically.


### Step 4.5 — Connect and verify via MCP Inspector

1. Click **Connect**
2. Inspector calls `http://localhost:8080/realms/mcp-realm/clients-registrations/openid-connect` → gets a new `client_id`
3. A browser window opens with the Keycloak login page
4. Log in with `testuser` / `password`
5. Inspector shows **"Successfully authenticated with OAuth"**

<img src="keycloak_screenshots/key_29.png" width="500">

<img src="keycloak_screenshots/key_30.png" width="500">


**Verify:**
1. **Tools** tab → **List Tools** — all 24 tools visible
2. Run a read tool → success
3. Run a write tool → success

<img src="keycloak_screenshots/key_32.png" width="500">

### Step 4.7 — Connect via VS Code

In VS Code, `ctrl + P` and run `MCP: Open User Configuration`, add the MCP server to `mcp.json`  with no token and no client ID — VS Code drives the full DCR flow automatically:

```json
{
  "servers": {
    "couchbase-keycloak-dcr": {
      "type": "http",
      "url": "http://127.0.0.1:8000/mcp"
    }
  }
}
```

On connect, VS Code will:
1. Hit the MCP server → get a `401`
2. Read the PRM document → discover Keycloak
3. Call the DCR endpoint → self-register → get a new `client_id`
4. Open a browser to Keycloak's `/authorize`
5. Log in with `testuser` / `password`
6. VS Code receives the JWT → connected


### Step 4.8 - Connect via Cursor

In Cursor, to open the MCP configuration file, go to settings and search for `Tools & MCPs`. Click on `New MCP Server`, this opens mcp.json. Add the following:

```json
{
  "mcpServers": {
    "couchbase-keycloak-dcr": {
      "type": "http",
      "url": "http://127.0.0.1:8000/mcp"
    }
  }
}
```

Go back to `Tools & MCPs` in the settings and click on `Connect` next to the couchbase MCP. Grant Access to Keycloak and complete the Authentication.

<img src="keycloak_screenshots/key_33.png" width="500">

<img src="keycloak_screenshots/key_34.png" width="500">

Ask the chat to use one of the MCP tools to validate.


<img src="keycloak_screenshots/key_35.png" width="500">
